In [ ]:
#2026-05-15
from apis_core.apis_metainfo.models import Uri, Collection, CollectionType
from apis_core.apis_vocabularies.models import ProfessionType, PersonInstitutionRelation
from apis_core.apis_entities.models import Person, Institution
from apis_core.apis_relations.models import PersonInstitution
from acdh_tei_pyutils.tei import TeiReader
from acdh_tei_pyutils.utils import any_xpath, get_xmlid, make_entity_label
from normdata.utils import import_from_normdata
from tqdm import tqdm

In [ ]:
col, _ = Collection.objects.get_or_create(name="Schönberg-Briefe: Universal-Edition und Dreililien")
domain = "schoenberg-ue"
col_type, _ = CollectionType.objects.get_or_create(name="Projekt")
col.description = """
Arnold Schönberg: Briefwechsel mit den Verlagen Universal-Edition und Dreililien. Hrsg. von Katharina Bleier und Therese Muxeneder unter Mitarbeit von Jannik Franz und Philipp Kehrer, Universität für Musik und darstellende Kunst Wien und Arnold Schönberg Center, Wien
<a href="https://www.schoenberg-ue.at">https://www.schoenberg-ue.at</a>
"""
col.collection_type = col_type
col.save()

In [ ]:
source_file_uri = "https://raw.githubusercontent.com/Schoenberg-UE/Schoenberg-Data/refs/heads/main/data/indices/Personen.xml"

In [ ]:
doc = TeiReader(source_file_uri)

In [ ]:
items = doc.any_xpath(".//tei:person[@xml:id and ./tei:persName[@type='reg']]")
url_stub = "https://www.schoenberg-ue.at/ue/persons/"
item_item_relation_type = PersonInstitutionRelation.objects.get(name="in Bezug zu") 


In [ ]:
print(len(items))
for x in tqdm(items, total=len(items)):
    try:
        xml_id = get_xmlid(x)
        tmp_uri = f"{url_stub}{xml_id}"
        try:
            first_name = any_xpath(x, "./tei:persName[@type='reg']/tei:forename[1]/text()")[0]
        except IndexError:
            first_name = None
        try:
            last_name = any_xpath(x, "./tei:persName[@type='reg']/tei:surname[1]/text()")[0]
        except IndexError:
            last_name = any_xpath(x, "./tei:persName[@type='alt'][0]/text()")[0]
        try:
            label = make_entity_label(any_xpath(x, "./tei:persName[@type='reg']")[0])[0]
        except IndexError:
            continue
        try:
            norm_data = any_xpath(x, "./tei:idno[@type='gnd']/text()")[0]
            entity = import_from_normdata(norm_data, "person")
            if entity:
                pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_uri, domain=domain)
                pmb_uri.entity = entity
                pmb_uri.save()
                entity.collection.add(col)
        except IndexError:
            pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_uri, domain=domain)
            entity = pmb_uri.entity
            if entity:
                pass
            else:
                item = {
                    "name": last_name,
                    "first_name": first_name
                }
                entity = Person.objects.create(**item)
                entity.collection.add(col)
                pmb_uri.entity = entity
                pmb_uri.save()
        person = Person.objects.get(id=entity.id)
        for y in any_xpath(x, ".//tei:occupation/tei:term"):
            prof = ProfessionType.objects.filter(name__startswith=y.text)
            person.profession.add(*prof)
        for y in any_xpath(x, ".//tei:affiliation/@key"):
            related_xml_id = y
            related_url = f"https://www.schoenberg-ue.at/ue/organisations/{related_xml_id}"
            related_entity = Uri.objects.get(uri=related_url).entity
            related_entity = Institution.objects.get(id=related_entity.id)
            person = Person.objects.get(id=entity.id)
            PersonInstitution.objects.get_or_create(related_person=person, related_institution=related_entity, relation_type=item_item_relation_type)
    except:  # noqa
        continue
     